# Pipeline обработки данных.

Чтение очищенного датасета. (датасет чистим в `eda.ipynb`)

## Задание 1 предобработка представлен в `eda.ipynb`

In [1]:
import pandas as pd

In [3]:
dataset_path = "dataset_clean.csv"

df = pd.read_csv(dataset_path, parse_dates=["TS"])

df.head()

,TS,Lab1_G1_N1,Lab1_G1_N2,Lab1_G1_N3,Lab1_G1_P2,Lab1_G1_T4ср,Lab1_G1_T1,Lab1_G1_T607,Lab1_G1_T600,Lab1_G1_T638,...,Lab1_AVOM_AVOMN1,Lab1_Kp,Lab1_hGPA,Lab1_Kran_5,Lab1_Kran_2,Lab1_Kran_6,Lab1_U_Kran_GPA_A_APK,Lab1_Kran_1,Lab1_Kran_4,Lab1_q
0,2022-12-02 18:59:59.999998+00:00,8726,11490,5000,15.35,662.7,-16.1,36.1,67.1,79.8,...,1,0.92,0.27,0,1,0,0,1,0,50.8
1,2022-12-02 19:00:59.999998+00:00,8726,11489,4993,15.35,662.4,-16.4,36.1,67.0,79.8,...,1,0.92,0.27,0,1,0,0,1,0,50.8
2,2022-12-02 19:01:59.999999+00:00,8725,11493,5001,15.35,663.1,-16.0,36.1,67.0,79.8,...,1,0.92,0.27,0,1,0,0,1,0,50.8
3,2022-12-02 19:02:59.999999+00:00,8726,11493,5011,15.34,663.0,-16.1,36.1,67.1,79.8,...,1,0.92,0.27,0,1,0,0,1,0,50.8
4,2022-12-02 19:04:00+00:00,8728,11496,5003,15.33,663.4,-15.9,36.1,67.1,80.0,...,1,0.92,0.27,0,1,0,0,1,0,50.8


## Задание 2. Расчет показателей Хёрста и Ляпунова.

In [ ]:
import numpy as np
import nolds

Подготовим вспомогательную функцию: ряд нормализуется. Константные и слишком короткие ряды не анализируются, потому что для них R/S-анализ и показатель Ляпунова неинформативны.

In [13]:
def prepare_series(series):
    x = series.to_numpy()

    if len(x) < 100:
        return None

    std = x.std()
    if std == 0:
        return None

    return (x - x.mean()) / std

Выберем числовые признаки с ненулевой динамикой. Временная колонка `TS` используется только для сортировки наблюдений.

In [14]:
df_tda = df.sort_values("TS").reset_index(drop=True)

numeric_cols = df_tda.select_dtypes(include="number").columns.tolist()
numeric_cols = [
    col for col in numeric_cols
    if df_tda[col].nunique(dropna=False) > 1
]

print(f"Рядов для анализа: {len(numeric_cols)}")

Рядов для анализа: 84


Для каждого ряда считаем показатель Хёрста методом R/S-анализа и крупнейший показатель Ляпунова методом Розенштейна (`lyap_r`).

*пояснение*:

Для оценки показателя Ляпунова был использован метод Розенштейна, поскольку он позволяет оценить крупнейший показатель
Ляпунова непосредственно по одномерному временному ряду без задания аналитической модели системы. Метод основан на
реконструкции фазового пространства с помощью временных задержек и последующем анализе скорости расхождения близких
траекторий. Положительное значение крупнейшего показателя Ляпунова интерпретируется как признак чувствительности к
начальным условиям и возможной хаотической динамики, тогда как неположительное значение не подтверждает наличие
хаотического поведения. Такой подход является практичным для экспериментальных и промышленных временных рядов, где
доступны только наблюдения датчиков, но неизвестны уравнения порождающего процесса.

In [ ]:
results = []

# Каждый числовой признак рассматриваем как отдельный одномерный временной ряд.
for col in numeric_cols:
    x = prepare_series(df_tda[col])

    # Если ряд слишком короткий или константный, показатели для него не считаются.
    if x is None:
        results.append({
            "column": col,
            "hurst_rs": np.nan,
            "lyapunov": np.nan,
            "error": "too short or constant series",
        })
        continue

    # Показатель Хёрста через R/S-анализ:
    # H < 0.5 - антиперсистентность, H ~= 0.5 - случайный процесс,
    # H > 0.5 - персистентность и наличие долгосрочной памяти.
    try:
        hurst = nolds.hurst_rs(
            x,
            fit="poly",
            corrected=True,
            unbiased=True,
        )
    except Exception as exc:
        hurst = np.nan
        hurst_error = str(exc)
    else:
        hurst_error = ""

    # Крупнейший показатель Ляпунова методом Розенштейна.
    # Положительное значение указывает на чувствительность к начальным условиям.
    try:
        lyapunov = nolds.lyap_r(
            x,
            emb_dim=6,
            lag=1,
            min_tsep=10,
            trajectory_len=20,
            fit="poly",
        )
    except Exception as exc:
        lyapunov = np.nan
        lyapunov_error = str(exc)
    else:
        lyapunov_error = ""

    # Сохраняем численные значения и возможные ошибки расчета для диагностики.
    results.append({
        "column": col,
        "hurst_rs": hurst,
        "lyapunov": lyapunov,
        "error": "; ".join(
            err for err in [hurst_error, lyapunov_error]
            if err
        ),
    })

| Колонка    | Пояснение                                                                                                   |
| ---------- | ----------------------------------------------------------------------------------------------------------- |
| `column`   | Название исследуемого признака (временного ряда / сенсора / параметра установки)                            |
| `hurst_rs` | Показатель Херста, характеризующий наличие долгосрочной памяти и степень персистентности временного ряда    |
| `lyapunov` | Оценка наибольшего показателя Ляпунова, характеризующая чувствительность динамики ряда к начальным условиям |
| `error`    | Текст ошибки вычисления метрик (если расчёт для признака завершился неуспешно)                              |


In [10]:
process_report = pd.DataFrame(results)
process_report

,column,hurst_rs,lyapunov,error
0,Lab1_G1_N1,0.896420,0.079421,
1,Lab1_G1_N2,0.892538,0.084435,
2,Lab1_G1_N3,0.485665,0.043871,
3,Lab1_G1_P2,0.744458,0.067160,
4,Lab1_G1_T4ср,0.885607,0.082348,
...,...,...,...,...
79,Lab1_TposleNag,0.882282,0.015593,
80,Lab1_PdoNag,1.037843,0.047536,
81,Lab1_TdoNag,1.071130,-0.003344,
82,Lab1_Rc,0.645156,0.021044,


Добавим качественную интерпретацию.

| Термин                  | Семантически                                            | Визуально                       | Показатель Херста        | Показатель Ляпунова          |
| ----------------------- | ------------------------------------------------------- | ------------------------------- | ------------------------ | ---------------------------- |
| **Персистентность**     | Ряд “помнит” прошлое: если рос, скорее продолжит расти  | плавные тренды, инерция         | $$H \in (0.5; 1]$$       | $$--$$                       |
| **Антиперсистентность** | Ряд часто меняет направление: после роста вероятен спад | зигзаги, возврат к среднему     | $$H \in [0; 0.5)$$       | $$--$$                       |
| **Случайный процесс**   | Долгосрочная память выражена слабо                      | шумоподобное поведение          | $$H \approx 0.5$$        |  $$\lambda \approx 0$$       |
| **Хаотичность**         | Малые изменения быстро приводят к разным траекториям    | сложное, нестабильное поведение | $$--$$                   | $$\lambda > 0$$              |


In [11]:
process_report["hurst_type"] = np.select(
    [
        process_report["hurst_rs"] < 0.45,
        process_report["hurst_rs"].between(0.45, 0.55),
        process_report["hurst_rs"] > 0.55,
    ],
    [
        "антиперсистентный процесс",
        "близко к случайному процессу",
        "персистентный процесс",
    ],
    default="не определено",
)

process_report["lyapunov_type"] = np.select(
    [
        process_report["lyapunov"] > 0,
        process_report["lyapunov"] <= 0,
    ],
    [
        "признаки хаотической динамики",
        "хаотичность не подтверждается",
    ],
    default="не определено",
)

display(process_report.sort_values("lyapunov", ascending=False))

,column,hurst_rs,lyapunov,error,hurst_type,lyapunov_type
44,Lab1_G3_Pc1,0.441834,0.194483,,антиперсистентный процесс,признаки хаотической динамики
45,Lab1_G3_Pc2,0.694835,0.149913,,персистентный процесс,признаки хаотической динамики
46,Lab1_G3_Pc3,0.556710,0.133554,,персистентный процесс,признаки хаотической динамики
47,Lab1_G3_T600,0.909729,0.130260,,персистентный процесс,признаки хаотической динамики
41,Lab1_G3_T638,0.903007,0.123888,,персистентный процесс,признаки хаотической динамики
...,...,...,...,...,...,...
73,Lab1_dPmg,0.667116,0.026149,,персистентный процесс,признаки хаотической динамики
74,Lab1_Pm_sm_N,0.610902,0.025586,,персистентный процесс,признаки хаотической динамики
82,Lab1_Rc,0.645156,0.021044,,персистентный процесс,признаки хаотической динамики
79,Lab1_TposleNag,0.882282,0.015593,,персистентный процесс,признаки хаотической динамики


Сводные статистики нужны для общего вывода о природе порождающих процессов по всем временным рядам.

In [12]:
display(process_report[["hurst_rs", "lyapunov"]].describe())
display(process_report["hurst_type"].value_counts().to_frame("count"))
display(process_report["lyapunov_type"].value_counts().to_frame("count"))

,hurst_rs,lyapunov
count,84.000000,84.000000
mean,0.744078,0.062014
std,0.175706,0.030784
min,0.388588,-0.003344
25%,0.606125,0.042886
50%,0.731952,0.053522
75%,0.874413,0.079299
max,1.362306,0.194483


,count
hurst_type,
персистентный процесс,72
близко к случайному процессу,10
антиперсистентный процесс,2


,count
lyapunov_type,
признаки хаотической динамики,83
хаотичность не подтверждается,1


### Выводы по показателям Хёрста и Ляпунова

По результатам расчета для 84 не константных временных рядов среднее значение показателя Хёрста составило `0.744`, медианное значение - `0.732`. Большинство рядов (`72` из `84`) относятся к персистентным процессам, то есть демонстрируют наличие долгосрочной памяти: текущее направление изменения с высокой вероятностью связано с предыдущей динамикой. Еще `10` рядов близки к случайному процессу, и только `2` ряда имеют признаки антиперсистентности.

Крупнейший показатель Ляпунова в среднем положителен (`0.062`), при этом положительные значения получены для `83` из `84` рядов. Это указывает на чувствительность большинства временных рядов к начальным условиям и может рассматриваться как эмпирический признак хаотической или сложной нелинейной динамики. В совокупности результаты позволяют сделать вывод, что порождающие процессы в данных преимущественно не являются простыми случайными процессами: для них характерны долгосрочная зависимость и признаки чувствительности к начальным условиям.

### Интерпретация результатов перед вложением в облако точек

Полученные значения показывают, что дальнейшее вложение временных рядов в облако точек является содержательно оправданным. Большинство рядов имеют показатель Хёрста выше `0.55` (`72` из `84`), то есть обладают персистентностью и долгосрочной зависимостью. Это означает, что соседние наблюдения во времени несут информацию о состоянии процесса. Поэтому при построении delay embedding последовательные лаги могут формировать осмысленную геометрию, а не случайное облако точек.

Почти все ряды имеют положительный крупнейший показатель Ляпунова (`83` из `84`), что указывает на чувствительность к начальным условиям и возможную нелинейную динамику. Для следующего этапа это означает, что фазовое облако точек может иметь нетривиальную структуру: близкие состояния процесса способны со временем расходиться, а значит топологический анализ такого облака может выявить особенности динамики, которые не видны напрямую на одномерном графике временного ряда. При этом результаты следует трактовать как эмпирическое обоснование выбора фазового вложения, а не как строгое доказательство хаотичности системы.